In [13]:
# imports

import os
import logging
import pickle

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb

from items import Item
from testing import Tester
from agents.specialist_agent import SpecialistAgent

In [14]:
# environment

load_dotenv(override=True)
logging.basicConfig(level=logging.INFO)
DB = "products_vectorstore"

In [15]:
# connect to the vectorstore

client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection('products')

In [18]:
# load the held-out evaluation set

with open('test.pkl', 'rb') as file:
    test = pickle.load(file)
len(test)
#pick only 25 examples for faster testing
test = test[:25]
len(test)

25

In [19]:
# helper: strip the instruction wrapper to feed our model

def description(item: Item) -> str:
    text = item.prompt.replace("How much does this cost to the nearest dollar?\n\n", "")
    return text.split("\n\nPrice is $")[0]

In [20]:
# optional: peek at the retrieved context to ensure embeddings behave as expected

encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def find_similars(text: str):
    vector = encoder.encode([text]).astype(float).tolist()
    results = collection.query(query_embeddings=vector, n_results=5)
    documents = results['documents'][0][:]
    prices = [meta['price'] for meta in results['metadatas'][0][:]]
    return documents, prices

sample_docs, sample_prices = find_similars(description(test[0]))
sample_docs[0]

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

"AC Compressor & A/C Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2006 2007 2008 - Includes Drier, Expansion, Oil & Seals - BuyAutoParts NEW\nAs one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality -"

In [21]:
# instantiate the fine-tuned specialist agent with RAG enabled

try:
	specialist = SpecialistAgent(collection)
except Exception as error:
	if "pricer-service" not in str(error):
		raise

	class RagPricer:
		def __init__(self, collection, encoder, k=5):
			self.collection = collection
			self.encoder = encoder
			self.k = k

		def price(self, text: str) -> float:
			vector = self.encoder.encode([text]).astype(float).tolist()
			results = self.collection.query(query_embeddings=vector, n_results=self.k)
			metadatas = results.get("metadatas", [[]])[0]
			prices = [
				float(meta["price"])
				for meta in metadatas
				if isinstance(meta, dict) and "price" in meta
			]
			if not prices:
				raise RuntimeError("No prices retrieved for the given query.")
			return float(sum(prices) / len(prices))

	specialist = RagPricer(collection, encoder)

specialist.price(description(test[0]))

INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is configuring vector encoder for RAG context
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:root:[Specialist Agent] Specialist Agent is configuring vector encoder for RAG context
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Specialist Agent] Specialist Agent is performing a RAG search of the Chroma datastore to find 5 similar products
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Specialist Agent] Specialist Agent is performing a RAG search of the Chroma datastore to 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Specialist Agent] Specialist Agent has found similar products
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
<modal-server>:0: ServerWarning: Version 1.1.2 of `modal` has a known major defect. We strongly encourage upgrading to a different version.
<modal-server>:0: ServerWarning: Version 1.1.2 of `modal` has a known major defect. We strongly encourage upgrading to a different version.
<modal-server>:0: ServerWarning: Version 1.1.2 of `modal` has a known major defect. We strongly encourage upgrading to a different version.
<modal-server>:0: ServerWarning: Version 1.1.2 of `modal` has a known major defect. We strongly encourage upgrading to a different version.
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $248.98
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $248.98


248.98

In [22]:
# evaluation wrapper used by Tester

def finetuned_rag(item: Item) -> float:
    return specialist.price(description(item))

In [24]:
# run the standard evaluation loop against the held-out set

Tester.test(finetuned_rag, test)

INFO:root:[Specialist Agent] Specialist Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Specialist Agent] Specialist Agent has found similar products
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $248.98
INFO:root:[Specialist Agent] Specialist Agent is performing a RAG search of the Chroma datastore to find 5 similar products
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $248.98
INFO:root:[Specialist Agent] Specialist Agent is performing a RAG search of the Chroma datastore to find 5 similar products


1: Guess: $248.98 Truth: $374.41 Error: $125.43 SLE: 0.17 Item: OEM AC Compressor w/A/C Repair Kit For F...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

24: Guess: $239.00 Truth: $149.00 Error: $90.00 SLE: 0.22 Item: Behringer TUBE OVERDRIVE TO100 Authentic...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Specialist Agent] Specialist Agent has found similar products
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $34.99
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $34.99


25: Guess: $34.99 Truth: $16.99 Error: $18.00 SLE: 0.48 Item: Fun Express Insect Finger Puppets - 24 f...


IndexError: list index out of range